# Pythia-1B с нуля на PyTorch

Этот ноутбук реализует архитектуру Pythia-1B / GPT-NeoX вручную на PyTorch.
`transformers` используется только для токенизатора, а не для самой модели.
Веса загружаются из официального репозитория `EleutherAI/pythia-1b`.

Что проверяется:
- структура модели и число параметров;
- загрузка весов без `AutoModelForCausalLM`;
- causal forward pass и KV-cache;
- perplexity на Tiny Shakespeare из `main.ipynb`;
- скорость prefill и autoregressive decoding.

Модель Pythia-1B имеет контекст 2048 токенов, 16 слоёв, hidden size 2048 и 8 attention heads.

In [17]:
# Если зависимости ещё не установлены, выполните в отдельной ячейке или в терминале:
# %pip install -U torch transformers huggingface_hub safetensors requests tqdm

import json
import math
import re
import time
from dataclasses import dataclass
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported():
    DTYPE = torch.bfloat16
elif DEVICE.type == 'cuda':
    DTYPE = torch.float16
else:
    DTYPE = torch.float32

print('device:', DEVICE)
print('dtype:', DTYPE)
print('torch:', torch.__version__)

device: cuda
dtype: torch.bfloat16
torch: 2.14.0+cu126


In [18]:
import gc

def clear_gpu_cache():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

## 1. Конфигурация Pythia-1B

Конфигурация соответствует `config.json` официальной модели. Pythia использует GPT-NeoX block с parallel residual: attention и MLP получают нормализованный вход параллельно и затем складываются с residual.

In [19]:
@dataclass
class PythiaConfig:
    vocab_size: int = 50304
    hidden_size: int = 2048
    intermediate_size: int = 8192
    num_hidden_layers: int = 16
    num_attention_heads: int = 8
    max_position_embeddings: int = 2048
    rotary_pct: float = 0.25
    rotary_emb_base: float = 10000.0
    layer_norm_eps: float = 1e-5
    hidden_act: str = 'gelu'
    use_parallel_residual: bool = True
    use_cache: bool = True
    attention_bias: bool = True

    @property
    def head_dim(self):
        assert self.hidden_size % self.num_attention_heads == 0
        return self.hidden_size // self.num_attention_heads

    @property
    def rotary_ndims(self):
        return int(self.head_dim * self.rotary_pct)

config = PythiaConfig()
print(config)
print('head_dim:', config.head_dim)
print('rotary_ndims:', config.rotary_ndims)

PythiaConfig(vocab_size=50304, hidden_size=2048, intermediate_size=8192, num_hidden_layers=16, num_attention_heads=8, max_position_embeddings=2048, rotary_pct=0.25, rotary_emb_base=10000.0, layer_norm_eps=1e-05, hidden_act='gelu', use_parallel_residual=True, use_cache=True, attention_bias=True)
head_dim: 256
rotary_ndims: 64


## 2. Rotary position embeddings

В Pythia/GPT-NeoX rotary embedding применяется только к первым 25% размерности attention head. Остальные координаты query и key проходят без rotary-преобразования.

In [20]:
def rotate_half(x):
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)


class RotaryEmbedding(nn.Module):
    def __init__(self, dim, base=10000.0):
        super().__init__()
        self.dim = dim
        self.base = base
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq, persistent=False)

    def forward(self, position_ids, dtype):
        # position_ids: [sequence_length]
        freqs = torch.outer(position_ids.float(), self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        cos = emb.cos().to(dtype=dtype)[None, None, :, :]
        sin = emb.sin().to(dtype=dtype)[None, None, :, :]
        return cos, sin


def apply_rotary(q, k, cos, sin, rotary_ndims):
    q_rot, q_pass = q[..., :rotary_ndims], q[..., rotary_ndims:]
    k_rot, k_pass = k[..., :rotary_ndims], k[..., rotary_ndims:]
    q_rot = q_rot * cos + rotate_half(q_rot) * sin
    k_rot = k_rot * cos + rotate_half(k_rot) * sin
    return torch.cat((q_rot, q_pass), dim=-1), torch.cat((k_rot, k_pass), dim=-1)

## 3. Attention, MLP и Transformer block

Здесь сохранены имена модулей GPT-NeoX. Благодаря этому state dict из Hugging Face можно загрузить напрямую, без ручного переименования каждого слоя.

In [21]:
class PythiaAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.num_attention_heads = config.num_attention_heads
        self.head_dim = config.head_dim
        self.rotary_ndims = config.rotary_ndims
        self.query_key_value = nn.Linear(
            config.hidden_size,
            3 * config.hidden_size,
            bias=config.attention_bias,
        )
        self.dense = nn.Linear(
            config.hidden_size,
            config.hidden_size,
            bias=config.attention_bias,
        )
        self.rotary_emb = RotaryEmbedding(config.rotary_ndims, config.rotary_emb_base)
        self.scale = self.head_dim ** -0.5

    def _causal_mask(self, query_length, key_length, past_length, device):
        # Строка i соответствует query с абсолютной позицией past_length + i.
        # Разрешены только ключи с позицией не больше позиции query.
        return torch.ones(query_length, key_length, device=device, dtype=torch.bool).tril(
            diagonal=past_length
        )

    def forward(self, hidden_states, past_key_value=None, use_cache=False):
        batch_size, query_length, _ = hidden_states.shape
        qkv = self.query_key_value(hidden_states)
        # GPT-NeoX layout: [batch, seq, heads, 3 * head_dim].
        # Нельзя использовать [batch, seq, 3, heads, head_dim]:
        # это перемешает Q/K/V между attention heads.
        qkv = qkv.view(
            batch_size, query_length, self.num_attention_heads, 3 * self.head_dim
        ).transpose(1, 2)
        query, key, value = qkv.chunk(3, dim=-1)

        past_length = 0 if past_key_value is None else past_key_value[0].shape[2]
        position_ids = torch.arange(
            past_length, past_length + query_length, device=hidden_states.device
        )
        cos, sin = self.rotary_emb(position_ids, hidden_states.dtype)
        query, key = apply_rotary(query, key, cos, sin, self.rotary_ndims)

        if past_key_value is not None:
            key = torch.cat((past_key_value[0], key), dim=2)
            value = torch.cat((past_key_value[1], value), dim=2)

        key_length = key.shape[2]
        causal_mask = self._causal_mask(
            query_length, key_length, past_length, hidden_states.device
        )
        attention_output = F.scaled_dot_product_attention(
            query, key, value,
            attn_mask=causal_mask,
            dropout_p=0.0,
            is_causal=False,
        )
        attention_output = attention_output.transpose(1, 2).contiguous()
        attention_output = attention_output.view(batch_size, query_length, -1)
        attention_output = self.dense(attention_output)

        present = (key, value) if use_cache else None
        return attention_output, present


class PythiaMLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.dense_h_to_4h = nn.Linear(
            config.hidden_size, config.intermediate_size, bias=True
        )
        self.dense_4h_to_h = nn.Linear(
            config.intermediate_size, config.hidden_size, bias=True
        )

    def forward(self, hidden_states):
        hidden_states = self.dense_h_to_4h(hidden_states)
        hidden_states = F.gelu(hidden_states)
        return self.dense_4h_to_h(hidden_states)


class PythiaDecoderLayer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.input_layernorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)
        self.post_attention_layernorm = nn.LayerNorm(
            config.hidden_size, eps=config.layer_norm_eps
        )
        self.attention = PythiaAttention(config)
        self.mlp = PythiaMLP(config)
        self.use_parallel_residual = config.use_parallel_residual

    def forward(self, hidden_states, past_key_value=None, use_cache=False):
        residual = hidden_states
        attention_input = self.input_layernorm(hidden_states)
        attention_output, present = self.attention(
            attention_input, past_key_value=past_key_value, use_cache=use_cache
        )

        if self.use_parallel_residual:
            mlp_input = self.post_attention_layernorm(hidden_states)
            mlp_output = self.mlp(mlp_input)
            hidden_states = residual + attention_output + mlp_output
        else:
            hidden_states = residual + attention_output
            hidden_states = hidden_states + self.mlp(
                self.post_attention_layernorm(hidden_states)
            )
        return hidden_states, present


class PythiaForCausalLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.gpt_neox = nn.Module()
        self.gpt_neox.embed_in = nn.Embedding(config.vocab_size, config.hidden_size)
        self.gpt_neox.layers = nn.ModuleList(
            [PythiaDecoderLayer(config) for _ in range(config.num_hidden_layers)]
        )
        self.gpt_neox.final_layer_norm = nn.LayerNorm(
            config.hidden_size, eps=config.layer_norm_eps
        )
        self.embed_out = nn.Linear(config.hidden_size, config.vocab_size, bias=False)

    def forward(self, input_ids, past_key_values=None, use_cache=False):
        hidden_states = self.gpt_neox.embed_in(input_ids)
        if past_key_values is None:
            past_key_values = [None] * len(self.gpt_neox.layers)

        presents = []
        for layer, past in zip(self.gpt_neox.layers, past_key_values):
            hidden_states, present = layer(
                hidden_states, past_key_value=past, use_cache=use_cache
            )
            if use_cache:
                presents.append(present)

        hidden_states = self.gpt_neox.final_layer_norm(hidden_states)
        logits = self.embed_out(hidden_states)
        return logits, tuple(presents) if use_cache else None

    @torch.no_grad()
    def generate_greedy(self, input_ids, max_new_tokens=64):
        self.eval()
        logits, past = self(input_ids, use_cache=True)
        generated = [input_ids]
        next_token = logits[:, -1:, :].argmax(dim=-1)
        generated.append(next_token)
        for _ in range(max_new_tokens - 1):
            logits, past = self(next_token, past_key_values=past, use_cache=True)
            next_token = logits[:, -1:, :].argmax(dim=-1)
            generated.append(next_token)
        return torch.cat(generated, dim=1)

## 4. Загрузка токенизатора и официальных весов

Мы не вызываем `AutoModelForCausalLM`. `huggingface_hub` скачивает файлы весов, `safetensors` читает их, а затем они загружаются в написанную выше модель.

In [22]:
from huggingface_hub import snapshot_download
from safetensors.torch import load_file as load_safetensors
from transformers import AutoTokenizer

MODEL_ID = 'EleutherAI/pythia-1b'
MODEL_DIR = Path(
    snapshot_download(
        repo_id=MODEL_ID,
        allow_patterns=[
            'config.json',
            'tokenizer*',
            '*.json',
            '*.safetensors',
            '*.bin',
        ],
    )
)

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), use_fast=True)
print('model directory:', MODEL_DIR)
print('tokenizer vocab:', tokenizer.vocab_size)
print('bos:', tokenizer.bos_token_id, 'eos:', tokenizer.eos_token_id)

def load_weight_file(path):
    if path.suffix == '.safetensors':
        return load_safetensors(str(path), device='cpu')
    try:
        return torch.load(path, map_location='cpu', weights_only=True)
    except TypeError:
        return torch.load(path, map_location='cpu')


def load_official_weights(model, model_dir):
    model_dir = Path(model_dir)
    index_candidates = [
        model_dir / 'model.safetensors.index.json',
        model_dir / 'pytorch_model.bin.index.json',
    ]
    state_dict = {}
    index_path = next((p for p in index_candidates if p.exists()), None)

    if index_path is not None:
        index = json.loads(index_path.read_text())
        shard_names = sorted(set(index['weight_map'].values()))
        for shard_name in shard_names:
            shard = load_weight_file(model_dir / shard_name)
            state_dict.update(shard)
            del shard
    else:
        single_candidates = [
            model_dir / 'model.safetensors',
            model_dir / 'pytorch_model.bin',
        ]
        weight_path = next((p for p in single_candidates if p.exists()), None)
        if weight_path is None:
            raise FileNotFoundError('Не найден файл весов в ' + str(model_dir))
        state_dict = load_weight_file(weight_path)

    # Некоторые версии GPT-NeoX сохраняют вспомогательные rotary buffers,
    # которых нет среди обучаемых параметров нашей реализации.
    expected_keys = set(model.state_dict().keys())
    ignored_keys = sorted(set(state_dict.keys()) - expected_keys)
    filtered_state_dict = {
        key: value for key, value in state_dict.items() if key in expected_keys
    }
    missing, unexpected = model.load_state_dict(filtered_state_dict, strict=False)
    del state_dict, filtered_state_dict
    if ignored_keys:
        print('ignored auxiliary checkpoint keys:', ignored_keys[:20])
    if missing:
        print('missing trainable keys:', missing[:20])
        print('checkpoint keys:', len(expected_keys))
        raise RuntimeError(
            'Не загружены обязательные параметры; проверьте списки missing keys'
        )
    if unexpected:
        print('unexpected keys after filtering:', unexpected[:20])
        raise RuntimeError('Неожиданные параметры в state dict')
    return model

model = PythiaForCausalLM(config)
model = load_official_weights(model, MODEL_DIR)
model = model.to(device=DEVICE, dtype=DTYPE)
model.eval()

parameter_count = sum(p.numel() for p in model.parameters())
print(f'parameters: {parameter_count:,} ({parameter_count / 1e9:.3f}B)')
print('model loaded successfully')

Fetching 6 files: 100%|██████████| 6/6 [00:00<00:00, 1042.88it/s]


model directory: /home/froschin/.cache/huggingface/hub/models--EleutherAI--pythia-1b/snapshots/f73d7dcc545c8bd326d8559c8ef84ffe92fea6b2
tokenizer vocab: 50254
bos: 0 eos: 0
ignored auxiliary checkpoint keys: ['gpt_neox.layers.0.attention.bias', 'gpt_neox.layers.0.attention.masked_bias', 'gpt_neox.layers.0.attention.rotary_emb.inv_freq', 'gpt_neox.layers.1.attention.bias', 'gpt_neox.layers.1.attention.masked_bias', 'gpt_neox.layers.1.attention.rotary_emb.inv_freq', 'gpt_neox.layers.10.attention.bias', 'gpt_neox.layers.10.attention.masked_bias', 'gpt_neox.layers.10.attention.rotary_emb.inv_freq', 'gpt_neox.layers.11.attention.bias', 'gpt_neox.layers.11.attention.masked_bias', 'gpt_neox.layers.11.attention.rotary_emb.inv_freq', 'gpt_neox.layers.12.attention.bias', 'gpt_neox.layers.12.attention.masked_bias', 'gpt_neox.layers.12.attention.rotary_emb.inv_freq', 'gpt_neox.layers.13.attention.bias', 'gpt_neox.layers.13.attention.masked_bias', 'gpt_neox.layers.13.attention.rotary_emb.inv_freq',

## 5. Проверка forward pass и KV-cache

Сначала прогоняем короткую последовательность. Затем проверяем, что logits последнего токена при cached decoding совпадают с logits dense-prefill в пределах погрешности выбранного dtype.

In [23]:
def max_abs_diff(a, b):
    return (a.float() - b.float()).abs().max().item()

with torch.inference_mode():
    test_ids = torch.randint(0, config.vocab_size, (1, 32), device=DEVICE)
    dense_logits, _ = model(test_ids, use_cache=False)
    prefill_logits, past = model(test_ids[:, :-1], use_cache=True)
    cached_logits, _ = model(
        test_ids[:, -1:], past_key_values=past, use_cache=True
    )

print('logits shape:', tuple(dense_logits.shape))
print('KV layers:', len(past))
print('KV shape in layer 0:', tuple(past[0][0].shape))
print('cached-vs-dense max abs diff:', max_abs_diff(
    dense_logits[:, -1:], cached_logits
))

logits shape: (1, 32, 50304)
KV layers: 16
KV shape in layer 0: (1, 8, 31, 256)
cached-vs-dense max abs diff: 0.09375


## 6. Tiny Shakespeare из `main.ipynb`

В `main.ipynb` используется датасет `tinyshakespeare/input.txt`. Следующая ячейка читает URL и имя файла из исходного ноутбука, поэтому benchmark остаётся привязанным к тому же источнику данных.

In [24]:
import requests

MAIN_NOTEBOOK = Path('/home/froschin/work/llm/main.ipynb')
WORK_DIR = MAIN_NOTEBOOK.parent

main_notebook = json.loads(MAIN_NOTEBOOK.read_text(encoding='utf-8'))
main_source = '\n'.join(
    ''.join(cell.get('source', []))
    for cell in main_notebook.get('cells', [])
)
# Эти значения совпадают с ячейкой загрузки датасета в main.ipynb.
SHAKESPEARE_URL = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
SHAKESPEARE_FILE = WORK_DIR / 'tinyshakespeare.txt'

if not SHAKESPEARE_FILE.exists():
    print('Downloading:', SHAKESPEARE_URL)
    response = requests.get(SHAKESPEARE_URL, timeout=60)
    response.raise_for_status()
    SHAKESPEARE_FILE.write_text(response.text, encoding='utf-8')

shakespeare_text = SHAKESPEARE_FILE.read_text(encoding='utf-8')
shakespeare_ids = tokenizer(
    shakespeare_text, add_special_tokens=False, return_tensors='pt'
).input_ids[0]

print('file:', SHAKESPEARE_FILE)
print('characters:', len(shakespeare_text))
print('tokens:', len(shakespeare_ids))
print('preview:', repr(shakespeare_text[:300]))

file: /home/froschin/work/llm/tinyshakespeare.txt
characters: 1115394
tokens: 340240
preview: "First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou are all resolved rather to die than to famish?\n\nAll:\nResolved. resolved.\n\nFirst Citizen:\nFirst, you know Caius Marcius is chief enemy to the people.\n\nAll:\nWe know't, we know't.\n\nFirst Citizen:\nLet us"


## 7. Benchmark: perplexity

Это benchmark pretrained-модели без дополнительного обучения на Shakespeare. Для честного сравнения Ocean нужно будет использовать те же token ids, те же окна и те же logits.

In [25]:
@torch.inference_mode()
def evaluate_perplexity(model, token_ids, max_tokens=8192, block_size=2048):
    token_ids = token_ids[:max_tokens].to(DEVICE)
    total_nll = 0.0
    total_tokens = 0
    model.eval()

    for start in range(0, len(token_ids) - 1, block_size):
        chunk = token_ids[start : start + block_size + 1]
        if len(chunk) < 2:
            continue
        input_ids = chunk[:-1].unsqueeze(0)
        targets = chunk[1:].unsqueeze(0)
        logits, _ = model(input_ids, use_cache=False)
        loss = F.cross_entropy(
            logits.float().reshape(-1, config.vocab_size),
            targets.reshape(-1),
            reduction='sum',
        )
        total_nll += loss.item()
        total_tokens += targets.numel()

    mean_nll = total_nll / total_tokens
    return mean_nll, math.exp(mean_nll)

ppl_start = time.perf_counter()
mean_nll, perplexity = evaluate_perplexity(model, shakespeare_ids)
if DEVICE.type == 'cuda':
    torch.cuda.synchronize()
ppl_elapsed = time.perf_counter() - ppl_start

print(f'mean NLL: {mean_nll:.4f}')
print(f'perplexity: {perplexity:.2f}')
print(f'evaluation time: {ppl_elapsed:.2f}s')

mean NLL: 2.9874
perplexity: 19.83
evaluation time: 1.78s


## 8. Benchmark: prefill и autoregressive decoding

`prefill` обрабатывает prompt целиком, а затем `generate_greedy` использует KV-cache и подаёт в модель по одному новому токену. Для Ocean именно второй участок удобно заменять routed attention.

In [26]:
def synchronize():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()

@torch.inference_mode()
def benchmark_generation(
    model, token_ids, prompt_length=512, new_tokens=64,
    allow_untrained_context=False,
):
    if prompt_length > config.max_position_embeddings:
        message = (
            f'prompt_length={prompt_length} превышает штатный контекст '
            f'{config.max_position_embeddings} токенов. Это экстраполяция RoPE, '
            'не проверенное расширение контекста.'
        )
        if not allow_untrained_context:
            raise ValueError(message + ' Передайте allow_untrained_context=True явно.')
        print('WARNING:', message)
    prompt = token_ids[:prompt_length].unsqueeze(0).to(DEVICE)
    model.eval()

    # Prefill измеряется отдельно: один dense forward по prompt.
    synchronize()
    prefill_start = time.perf_counter()
    logits, past = model(prompt, use_cache=True)
    synchronize()
    prefill_elapsed = time.perf_counter() - prefill_start

    next_token = logits[:, -1:, :].argmax(dim=-1)
    generated = [prompt, next_token]

    # Здесь уже есть KV-cache; измеряем только последующие one-token forwards.
    synchronize()
    decode_start = time.perf_counter()
    for _ in range(max(0, new_tokens - 1)):
        logits, past = model(next_token, past_key_values=past, use_cache=True)
        next_token = logits[:, -1:, :].argmax(dim=-1)
        generated.append(next_token)
    synchronize()
    decode_elapsed = time.perf_counter() - decode_start

    output = torch.cat(generated, dim=1)
    total_elapsed = prefill_elapsed + decode_elapsed
    decode_steps = max(1, new_tokens - 1)
    generated_text = tokenizer.decode(output[0].tolist())
    return {
        'output': generated_text,
        'prefill_seconds': prefill_elapsed,
        'prefill_tokens_per_second': prompt_length / prefill_elapsed,
        'decode_seconds': decode_elapsed,
        'decode_tokens_per_second': decode_steps / decode_elapsed,
        'total_seconds': total_elapsed,
        'total_tokens_per_second': new_tokens / total_elapsed,
    }

In [27]:
# benchmark = benchmark_generation(
#     model, shakespeare_ids, prompt_length=512, new_tokens=64
# )
# print(f'prefill seconds: {benchmark["prefill_seconds"]:.3f}')
# print(f'prefill tok/s: {benchmark["prefill_tokens_per_second"]:.2f}')
# print(f'decode seconds: {benchmark["decode_seconds"]:.3f}')
# print(f'decode tok/s: {benchmark["decode_tokens_per_second"]:.2f}')
# print(f'total seconds: {benchmark["total_seconds"]:.3f}')

# clear_gpu_cache()

prefill seconds: 0.107

prefill tok/s: 4790.68

decode seconds: 0.793

decode tok/s: 79.44

total seconds: 0.900

In [28]:
# benchmark = benchmark_generation(
#     model, shakespeare_ids, prompt_length=14_000, new_tokens=64,
#     allow_untrained_context=True,  # экспериментальная RoPE-экстраполяция
# )
# print(f'prefill seconds: {benchmark["prefill_seconds"]:.3f}')
# print(f'prefill tok/s: {benchmark["prefill_tokens_per_second"]:.2f}')
# print(f'decode seconds: {benchmark["decode_seconds"]:.3f}')
# print(f'decode tok/s: {benchmark["decode_tokens_per_second"]:.2f}')
# print(f'total seconds: {benchmark["total_seconds"]:.3f}')

# clear_gpu_cache()

prefill seconds: 6.845

prefill tok/s: 2045.15

decode seconds: 1.678

decode tok/s: 37.54

total seconds: 8.523

## 9. Ocean: routed attention для Pythia

Ниже создаётся вторая модель с теми же весами, но с заменённым attention-механизмом. В первой версии оптимизация включается только при autoregressive decoding после prefill. Это позволяет не нарушать causal semantics prefill.

Маршрут для каждого слоя и головы состоит из:
- 2 последних локальных блоков;
- 5 semantic-блоков с максимальной cosine similarity к среднему последних ключей;
- 1 deterministic exploration-блока.

Полный KV-cache сохраняется, но attention вычисляется только по выбранным блокам.

In [29]:
class OceanPythiaAttention(PythiaAttention):
    def __init__(
        self, config, layer_idx, block_size=64, summary_window=100,
        route_refresh_interval=50, local_blocks=2, semantic_blocks=5,
    ):
        super().__init__(config)
        self.layer_idx = layer_idx
        self.block_size = block_size
        self.summary_window = summary_window
        self.route_refresh_interval = route_refresh_interval
        self.local_blocks = local_blocks
        self.semantic_blocks = semantic_blocks
        self._route_cache = None
        self._route_num_blocks = None
        self._route_age = 0
        self._route_refresh_count = 0

    def reset_route(self):
        self._route_cache = None
        self._route_num_blocks = None
        self._route_age = 0
        self._route_refresh_count = 0

    def _dense_attention(self, query, key, value, past_length):
        query_length = query.shape[2]
        key_length = key.shape[2]
        mask = self._causal_mask(
            query_length, key_length, past_length, query.device
        )
        output = F.scaled_dot_product_attention(
            query, key, value, attn_mask=mask, dropout_p=0.0, is_causal=False
        )
        return output

    def _build_route(self, key):
        # Prototype рассчитан на batch=1; route различается по attention heads.
        batch_size, num_heads, key_length, _ = key.shape
        if batch_size != 1:
            raise NotImplementedError('Ocean prototype пока поддерживает только batch=1')

        num_blocks = math.ceil(key_length / self.block_size)
        padded_length = num_blocks * self.block_size
        pad = padded_length - key_length
        if pad:
            padded_key = F.pad(key, (0, 0, 0, pad))
        else:
            padded_key = key
        blocks = padded_key.view(1, num_heads, num_blocks, self.block_size, self.head_dim)
        counts = torch.full(
            (num_blocks,), self.block_size, device=key.device, dtype=key.dtype
        )
        if pad:
            counts[-1] = self.block_size - pad
        summaries = blocks.sum(dim=3) / counts.view(1, 1, num_blocks, 1)

        recent_start = max(0, key_length - self.summary_window)
        recent = key[:, :, recent_start:key_length, :].mean(dim=2)
        # recent: [1, heads, D], summaries: [1, heads, blocks, D].
        # Явное суммирование по D сохраняет отдельный score для каждого head/block.
        recent_norm = F.normalize(recent.float(), dim=-1).unsqueeze(2)
        summary_norm = F.normalize(summaries.float(), dim=-1)
        scores = (recent_norm * summary_norm).sum(dim=-1)

        local_start = max(0, num_blocks - self.local_blocks)
        local_ids = list(range(local_start, num_blocks))
        semantic_count = min(self.semantic_blocks, max(0, num_blocks - len(local_ids)))
        routes = []
        for head in range(num_heads):
            # stable=True сохраняет меньший block id при одинаковых score.
            order = torch.argsort(
                scores[0, head], descending=True, stable=True
            ).tolist()
            semantic_ids = [
                block_id for block_id in order if block_id not in local_ids
            ][:semantic_count]
            selected = local_ids + semantic_ids

            # Детерминированная псевдослучайная exploration без дубликатов.
            seed = (
                (self.layer_idx + 1) * 1000003
                + (self._route_refresh_count + 1) * 9176
                + head * 101
            )
            start = seed % num_blocks
            for offset in range(num_blocks):
                candidate = (start + offset) % num_blocks
                if candidate not in selected:
                    selected.append(candidate)
                    break
            routes.append(selected)

        route = torch.tensor(routes, device=key.device, dtype=torch.long)
        self._route_cache = route.unsqueeze(0)
        self._route_num_blocks = num_blocks
        self._route_age = 0
        self._route_refresh_count += 1
        return self._route_cache

    def _routed_attention(self, query, key, value):
        key_length = key.shape[2]
        num_blocks = math.ceil(key_length / self.block_size)
        if (
            self._route_cache is None
            or self._route_age >= self.route_refresh_interval
            or self._route_num_blocks != num_blocks
        ):
            route = self._build_route(key)
        else:
            route = self._route_cache
            self._route_age += 1

        # route: [batch=1, heads, route_blocks]. Собираем токены блоков.
        block_offsets = torch.arange(self.block_size, device=key.device)
        token_ids = route[:, :, :, None] * self.block_size + block_offsets
        valid = token_ids < key_length
        token_ids = token_ids.clamp(max=key_length - 1)
        route_tokens = token_ids.shape[2] * token_ids.shape[3]
        token_ids = token_ids.reshape(1, self.num_attention_heads, route_tokens)
        valid = valid.reshape(1, self.num_attention_heads, route_tokens)

        gather_ids = token_ids.unsqueeze(-1).expand(
            -1, -1, -1, self.head_dim
        )
        key_route = key.gather(2, gather_ids)
        value_route = value.gather(2, gather_ids)
        route_mask = valid[:, :, None, :]
        return F.scaled_dot_product_attention(
            query, key_route, value_route,
            attn_mask=route_mask, dropout_p=0.0, is_causal=False
        )

    def forward(self, hidden_states, past_key_value=None, use_cache=False):
        batch_size, query_length, _ = hidden_states.shape
        qkv = self.query_key_value(hidden_states)
        qkv = qkv.view(
            batch_size, query_length, self.num_attention_heads, 3 * self.head_dim
        ).transpose(1, 2)
        query, key, value = qkv.chunk(3, dim=-1)

        past_length = 0 if past_key_value is None else past_key_value[0].shape[2]
        position_ids = torch.arange(
            past_length, past_length + query_length, device=hidden_states.device
        )
        cos, sin = self.rotary_emb(position_ids, hidden_states.dtype)
        query, key = apply_rotary(query, key, cos, sin, self.rotary_ndims)

        if past_key_value is not None:
            key = torch.cat((past_key_value[0], key), dim=2)
            value = torch.cat((past_key_value[1], value), dim=2)

        # Prefill остаётся dense; sparse route включается только для q_len=1.
        if past_key_value is None or query_length != 1:
            attention_output = self._dense_attention(query, key, value, past_length)
            self.reset_route()
        else:
            attention_output = self._routed_attention(query, key, value)

        attention_output = attention_output.transpose(1, 2).contiguous()
        attention_output = attention_output.view(batch_size, query_length, -1)
        attention_output = self.dense(attention_output)
        present = (key, value) if use_cache else None
        return attention_output, present


def make_ocean_model(dense_model):
    ocean_model = PythiaForCausalLM(dense_model.config)
    for layer_idx, layer in enumerate(ocean_model.gpt_neox.layers):
        layer.attention = OceanPythiaAttention(
            dense_model.config, layer_idx=layer_idx, block_size=64,
            summary_window=100, route_refresh_interval=50,
            local_blocks=2, semantic_blocks=5,
        )
    ocean_model.load_state_dict(dense_model.state_dict(), strict=True)
    ocean_model = ocean_model.to(device=DEVICE, dtype=DTYPE)
    ocean_model.eval()
    return ocean_model


def reset_ocean_routes(ocean_model):
    for layer in ocean_model.gpt_neox.layers:
        layer.attention.reset_route()


ocean_model = make_ocean_model(model)
print('Ocean model created; weights copied from dense model')

Ocean model created; weights copied from dense model


In [30]:
# # Сравнение на 14K prompt. Это benchmark скорости, а не проверка качества
# # за пределами штатного контекста Pythia-1B.
# reset_ocean_routes(ocean_model)
# ocean_benchmark = benchmark_generation(
#     ocean_model, shakespeare_ids, prompt_length=14_000, new_tokens=64,
#     allow_untrained_context=True,
# )

# print(f'prefill seconds: {ocean_benchmark["prefill_seconds"]:.3f}')
# print(f'prefill tok/s: {ocean_benchmark["prefill_tokens_per_second"]:.2f}')
# print(f'Ocean decode seconds: {ocean_benchmark["decode_seconds"]:.3f}')
# print(f'Ocean decode tok/s: {ocean_benchmark["decode_tokens_per_second"]:.2f}')
# print(f'total seconds: {ocean_benchmark["total_seconds"]:.3f}')

# clear_gpu_cache()

prefill seconds: 6.855

prefill tok/s: 2042.38

Ocean decode seconds: 1.009

Ocean decode tok/s: 62.46

total seconds: 7.863

### Ограничения текущего Ocean prototype

1. Prefill пока dense, поэтому оптимизируется только decode.
2. Summary blocks пересчитываются при refresh маршрута полным проходом по KV-cache; hierarchy ещё не добавлена.
3. Реализован только batch=1.
4. Полный KV-cache сохраняется, то есть оптимизация уменьшает вычисления attention, но не память.
5. Для prompt длиннее 2048 качество Pythia не является валидным показателем без отдельного context-extension обучения.

Именно этот вариант предназначен как прозрачная точка старта для дальнейшей замены `_build_route` на иерархический selector и для переноса routing в causal prefill.

## 10. Perplexity: dense vs Ocean

Обычный benchmark perplexity обрабатывает окно целиком и поэтому не включает routed decoding. Здесь каждый токен подаётся отдельно через KV-cache: dense-модель просматривает полный cache, Ocean — выбранные блоки.

Тест ограничен 2048 токенами — штатным контекстом Pythia-1B.

In [31]:
@torch.inference_mode()
def evaluate_cached_perplexity(model, token_ids, max_tokens=2048):
    ids = token_ids[:max_tokens].to(DEVICE)
    if ids.numel() < 2:
        raise ValueError('Для perplexity нужно минимум два токена')

    if hasattr(model.gpt_neox.layers[0].attention, 'reset_route'):
        reset_ocean_routes(model)

    model.eval()
    synchronize()
    start = time.perf_counter()

    # Первый token создаёт KV-cache и logits для следующего token.
    logits, past = model(ids[:1].view(1, 1), use_cache=True)
    total_nll = 0.0
    total_targets = 0

    for index in range(1, ids.numel()):
        target = ids[index].view(1)
        log_probs = F.log_softmax(logits[:, -1, :].float(), dim=-1)
        total_nll += -log_probs[0, target].item()
        total_targets += 1

        if index < ids.numel() - 1:
            logits, past = model(
                ids[index:index + 1].view(1, 1),
                past_key_values=past,
                use_cache=True,
            )

    synchronize()
    elapsed = time.perf_counter() - start
    mean_nll = total_nll / total_targets
    return {
        'mean_nll': mean_nll,
        'perplexity': math.exp(mean_nll),
        'seconds': elapsed,
        'tokens_per_second': total_targets / elapsed,
        'tokens': total_targets,
    }

PPL_TOKENS = min(config.max_position_embeddings, 2048)
dense_ppl = evaluate_cached_perplexity(model, shakespeare_ids, PPL_TOKENS)
ocean_ppl = evaluate_cached_perplexity(ocean_model, shakespeare_ids, PPL_TOKENS)

print('--- dense ---')
print(f'tokens: {dense_ppl["tokens"]}')
print(f'mean NLL: {dense_ppl["mean_nll"]:.4f}')
print(f'perplexity: {dense_ppl["perplexity"]:.2f}')
print(f'time: {dense_ppl["seconds"]:.3f}s')
print(f'tok/s: {dense_ppl["tokens_per_second"]:.2f}')

print('--- Ocean ---')
print(f'tokens: {ocean_ppl["tokens"]}')
print(f'mean NLL: {ocean_ppl["mean_nll"]:.4f}')
print(f'perplexity: {ocean_ppl["perplexity"]:.2f}')
print(f'time: {ocean_ppl["seconds"]:.3f}s')
print(f'tok/s: {ocean_ppl["tokens_per_second"]:.2f}')

print('--- delta ---')
print(f'PPL delta: {ocean_ppl["perplexity"] - dense_ppl["perplexity"]:+.2f}')
print(f'NLL delta: {ocean_ppl["mean_nll"] - dense_ppl["mean_nll"]:+.4f}')
print(f'speedup: {dense_ppl["seconds"] / ocean_ppl["seconds"]:.2f}x')

--- dense ---
tokens: 2047
mean NLL: 3.0701
perplexity: 21.54
time: 28.904s
tok/s: 70.82
--- Ocean ---
tokens: 2047
mean NLL: 3.4758
perplexity: 32.33
time: 30.767s
tok/s: 66.53
--- delta ---
PPL delta: +10.78
NLL delta: +0.4057
speedup: 0.94x


--- dense ---

tokens: 2047

mean NLL: 3.0701

perplexity: 21.54

time: 28.904s

tok/s: 70.82

--- Ocean ---

tokens: 2047

mean NLL: 3.4758

perplexity: 32.33

time: 30.767s

tok/s: 66.53

--- delta ---

PPL delta: +10.78

NLL delta: +0.4057

speedup: 0.94x